# Tutorial 21: Magnetic SAXS through a 50 µm square window

Simulate a **1300 × 1300 detector, 20 µm pixels, 40 cm distance**, with uniform, coherent, vertically linearly polarized illumination at **778 eV** near the Co L₃ edge. The corrected stack is **SiN(100)/Ta(5)/Pt(3)/[Co(2)/Pt(3)/Al(1.4)]×15/Pt(5)** (thicknesses nm). The magnetic period is **200 nm**, meaning alternating 100 nm up/down domains. We assume perpendicular magnetization, identical through all Co layers, with straight walls parallel to y.

The 50 × 50 µm square is rotated **45°** about the beam axis; stripes remain parallel to laboratory y. It is the clear, film-spanned opening. Everything outside it is perfectly opaque. This replaces the 0.5 mm SiO₂ support and its 45° tapered walls with a thin absorbing boundary: it omits wall scattering, channel propagation and angle-dependent vignetting. If 50 µm describes the *wide* end of the taper, specify the narrow clear opening instead. No fictitious thick gold layer is needed.

We use the standard `ExperimentConfig` and production Jones solver to obtain the two local domain transmissions. A Fourier-series expansion of the stripes and analytic rotated-square aperture integrals then supplies the full physical detector. This specialized detector calculation avoids allocating a 50 µm-wide nanometre-grid multilayer volume. It assumes projection through the 209 nm film, abrupt domain walls, normal incidence and no interlayer diffraction/reflection. It is not a general multislice solver for arbitrary textures.


In [1]:
from pathlib import Path
import sys, json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from scipy.special import fresnel
import h5py
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/scattering_calculator").is_dir())
sys.path.insert(0, str(ROOT / "src"))
from scattering_calculator import simulation_pipelines as sim
OUT = ROOT / "outputs/21_square_window_saxs"
OUT.mkdir(parents=True, exist_ok=True)


## 1. Declare the source, film and camera

The 2 × 2 grid is a **local transmission probe**, with one up and one down domain per row; it is not the full aperture sampling. Unit photon flux is a reference. Set a measured flux through the clear square to predict ideal photon rates. No detector efficiency, noise, beamstop or analyzer is applied. Vertical incident polarization is +y; the crossed component is +x.


In [2]:
window_size = 50e-6
period = 200e-9
window_angle = np.deg2rad(45.)
footprint_samples = 5  # midpoint samples per pixel axis; compare with 9 below
max_harmonic = 15  # odd stripe Fourier orders; compare with 25 below
experiment = sim.ExperimentConfig(
    xray=sim.XRayConfig(energy=778., photon_flux=1., pol="LV"),
    simulation=sim.SimulationConfig(shape=(2, 2), real_space_pixel_size=period/2),
    sample=sim.SampleConfig(recipe="SiN(100)/Ta(5)/Pt(3)/[Co(2)/Pt(3)/Al(1.4)]x15/Pt(5)",
        sample_name="SiN-supported magnetic stripe window"),
    magnetic_pattern=sim.MagneticPatternConfig(pattern_type_method="saturated_pattern"),
    aperture=sim.FrontApertureConfig(aperture_method=None),
    illumination=sim.IlluminationConfig(illumination_function=None),
    propagation=sim.SamplePropagatorConfig(propagator_method="Jones",
        propagator_config={"propagate":False}),
    detector=sim.DetectorConfig(shape=(1300,1300), detector_center=(649.5,649.5),
        pixel_size=20e-6, sample_to_detector_distance=.40,
        detector_propagation_method="fraunhofer", analyzer_angle=None),
    beamstop=sim.BeamstopConfig(bs_method=None),
    outputs=sim.OutputConfig(path=OUT / "local_transmission.h5", overwrite=True,
        save=("exit_wave",)),
    analysis={"window_size_m":window_size, "window_angle_rad":window_angle, "stripe_period_m":period,
              "support_model":"opaque exterior; tapered walls omitted",
              "detector_integrator":"rotated aperture Fourier-series Fresnel and Fraunhofer integrals"},
)
# The detector selector above describes the production FFT option only.
# The full-window detector below is computed explicitly, including Fresnel phase.
m = np.zeros((2,2,3)); m[...,2] = [[1,-1],[1,-1]]
results = sim.simulate_experiment(experiment, magnetization=m)
# Four equal probe pixels have incident amplitude sqrt(flux/4).
transmission = results.read("exit_wave")[0] / np.sqrt(experiment.xray.photon_flux/4)
t_plus, t_minus = transmission
wavelength = experiment.xray.setup().wavelength
z = experiment.detector.sample_to_detector_distance
pitch = experiment.detector.pixel_size
print("Domain Jones transmissions (Ex,Ey):", transmission)
print(f"Wavelength: {wavelength*1e9:.4f} nm; film thickness: 209 nm")
print(f"Fresnel number L²/(λz): {window_size**2/(wavelength*z):.3f}")
first_radius = z*np.tan(np.arcsin(wavelength/period))
print(f"First satellites: ±{first_radius*1e3:.3f} mm, ±{first_radius/pitch:.1f} pixels")
print(f"Far-field aperture fringe spacing: {wavelength*z/window_size*1e6:.2f} µm")


Domain Jones transmissions (Ex,Ey): [[-0.02220814-0.00127461j  0.53344818+0.05835542j]
 [ 0.02220814+0.00127461j  0.53344818+0.05835542j]]
Wavelength: 1.5936 nm; film thickness: 209 nm
Fresnel number L²/(λz): 3.922
First satellites: ±3.187 mm, ±159.4 pixels
Far-field aperture fringe spacing: 12.75 µm


## 2. Rotated aperture and finite-distance diffraction

The square has local coordinates $u=x\cos\alpha+y\sin\alpha$ and $v=-x\sin\alpha+y\cos\alpha$, with $\alpha=45°$ and $|u|,|v|<L/2$. The stripes remain fixed along laboratory y: $m_z(x)=\mathrm{sign}(\cos(2\pi x/\Lambda))$. A 200 nm period still means two 100 nm domains. Rotating the window does not rotate the stripes or incident polarization.

Expand the Jones transmission as $t(x)=t_0+\Delta t\sum_h c_h e^{i2\pi hx/\Lambda}$, where $t_0=(t_++t_-)/2$, $\Delta t=(t_+-t_-)/2$ and, for positive odd h, $c_{\pm h}=2(-1)^{(h-1)/2}/(\pi h)$. Each order is a product of two exact interval integrals in the rotated square axes. Fresnel retains the source quadratic phase, while Fraunhofer drops it. Completing the square gives shifted Fresnel integrals for each harmonic; amplitudes from all orders are combined **coherently** before summing transverse component powers.

The normalization is $\sqrt{\Phi}/(L\lambda z)$, so pixel-area-integrated intensity has units photons/s. Midpoint footprint samples average intensity within each camera pixel. Both models are paraxial; neither includes tapered-wall effects. Harmonic truncation and pixel integration are checked below.

For speed, the dephased one-dimensional Fresnel aperture envelope is cached with cubic interpolation at 0.1 µm spacing. The common quadratic phase is restored after combining orders. The independent quadrature check below also checks this interpolation.


In [3]:
ny, nx = experiment.detector.shape
cy, cx = experiment.detector.detector_center
x_detector = (np.arange(nx)-cx)*pitch
y_detector = (np.arange(ny)-cy)*pitch
mean_t = (t_plus+t_minus)/2
delta_t = (t_plus-t_minus)/2

def aperture_integral(points, method):
    a, b = -window_size/2, window_size/2
    if method == "fresnel":
        scale = np.sqrt(2/(wavelength*z))
        sa, ca = fresnel((a-points)*scale)
        sb, cb = fresnel((b-points)*scale)
        return np.sqrt(wavelength*z/2)*((cb-ca)-1j*(sb-sa))
    if method == "fraunhofer":
        return window_size*np.sinc(points*window_size/(wavelength*z))
    raise ValueError(method)

# Remove the common detector quadratic phase before interpolation. The
# remaining aperture envelope varies on the ~12.75 µm fringe scale.
# Cache it at 0.1 µm spacing to accelerate the full detector footprint sum.
lookup_pitch = .1e-6
lookup_limit = .10
lookup_x = np.arange(-lookup_limit,lookup_limit+lookup_pitch/2,lookup_pitch)
lookup_envelope = aperture_integral(lookup_x,"fresnel")*np.exp(1j*np.pi*lookup_x**2/(wavelength*z))
def fresnel_envelope(points):
    position = (points-lookup_x[0])/lookup_pitch
    index = np.floor(position).astype(np.int64)
    if np.any(index < 1) or np.any(index >= len(lookup_x)-2):
        raise ValueError("Increase lookup_limit for this geometry/harmonic count")
    fraction = position-index
    # Cubic interpolation on a uniform grid avoids repeated interval searches.
    a, b = lookup_envelope[index-1],lookup_envelope[index]
    c, d = lookup_envelope[index+1],lookup_envelope[index+2]
    return b+.5*fraction*(c-a+fraction*(2*a-5*b+4*c-d+fraction*(3*(b-c)+d-a)))

def rotated_field(xx, yy, method, cutoff):
    c, s = np.cos(window_angle), np.sin(window_angle)
    u, v = c*xx+s*yy, -s*xx+c*yy
    integral = fresnel_envelope if method == "fresnel" else lambda p: aperture_integral(p,method)
    base = integral(u)*integral(v)
    modulation = np.zeros_like(base,dtype=complex)
    for h in range(1,cutoff+1,2):
        coefficient = 2*(-1)**((h-1)//2)/(np.pi*h)
        for order in (-h,h):
            fu, fv = order*c/period, -order*s/period
            shifted_u, shifted_v = u+wavelength*z*fu, v+wavelength*z*fv
            term = integral(shifted_u)*integral(shifted_v)
            modulation += coefficient*term
    # Restore the common phase for independent complex-field checks.
    field = base[...,None]*mean_t + modulation[...,None]*delta_t
    if method == "fresnel":
        field *= np.exp(-1j*np.pi*(u*u+v*v)/(wavelength*z))[...,None]
    return field

def detector_pattern(samples=5, method="fresnel", cutoff=None):
    cutoff = max_harmonic if cutoff is None else cutoff
    offsets = ((np.arange(samples)+.5)/samples-.5)*pitch
    components = np.zeros((ny,nx,2))
    # Block rows to bound temporary complex arrays; no full specimen volume.
    for start in range(0,ny,32):
        stop = min(start+32,ny)
        for dy in offsets:
            for dx in offsets:
                field = rotated_field(x_detector[None,:]+dx,y_detector[start:stop,None]+dy,method,cutoff)
                components[start:stop] += abs(field)**2/samples**2
    components *= experiment.xray.photon_flux*pitch**2/(window_size*wavelength*z)**2
    return {"total":components.sum(axis=-1), "crossed":components[...,0],
            "parallel":components[...,1]}

patterns = detector_pattern(footprint_samples)
farfield = detector_pattern(footprint_samples,"fraunhofer")


## 3. Inspect the detector and magnetic satellites

All images use one normalization. The crossed image can be much weaker than the direct transmitted beam. The profile sums rows to retain photons/pixel-column rather than selecting a single potentially dark fringe. The ±1 vertical lines mark the nonparaxial grating prediction; Fresnel peak coordinates use its small-angle approximation. Odd higher harmonics arise from the abrupt square-wave domains.


In [4]:
%matplotlib qt

extent = ((x_detector[0]-pitch/2)*1e3,(x_detector[-1]+pitch/2)*1e3,
          (y_detector[0]-pitch/2)*1e3,(y_detector[-1]+pitch/2)*1e3)
peak = patterns["total"].max()
fig, axes = plt.subplots(1,3,figsize=(15,4.5),layout="constrained")
for ax, key in zip(axes,("total","parallel","crossed")):
    im = ax.imshow(np.maximum(patterns[key]/peak,1e-16), origin="lower",extent=extent,
        cmap="magma",norm=LogNorm(1e-10,1))
    ax.set(title=key, xlabel="Detector x (mm)",ylabel="Detector y (mm)")
    for sign in (-1,1):
        ax.plot(sign*first_radius*1e3,0,"o",mfc="none",mec="cyan")
fig.colorbar(im,ax=axes,label="Intensity / total detector peak")
fig.savefig(OUT / "detector.png",dpi=160)
plt.show()
fig, axes = plt.subplots(1,2,figsize=(13,4),layout="constrained")
for key in ("total","parallel","crossed"):
    axes[0].semilogy(x_detector*1e3,np.maximum(patterns[key].sum(axis=0),1e-25),label=key)
for model, image in (("Fresnel",patterns["total"]),("Fraunhofer",farfield["total"])):
    axes[1].semilogy(x_detector*1e3,np.maximum(image.sum(axis=0),1e-25),label=model)
for ax in axes:
    for sign in (-1,1): ax.axvline(sign*first_radius*1e3,color="gray",ls=":")
    ax.set(xlabel="Detector x (mm)",ylabel="Photons/s per pixel column")
    ax.legend()
axes[1].set_xlim(-.15,.15)
fig.savefig(OUT / "profiles.png",dpi=160)
#plt.show()


## 4. Numerical checks and saved output

Compare 5 versus 9 footprint samples per axis and 15 versus 25 stripe harmonics in the bright ±0.5 mm detector strip. This keeps refinement practical; it does not bound errors in faint tails outside the strip. Increase the settings if the integrated discrepancies exceed 1%. Verify the rotated clear-window area, compare the Fresnel field with independent midpoint quadrature in both square axes at a magnetic satellite, and locate the first crossed peak. Saved maps contain unnormalized ideal photon rates. The production local-transmission HDF5 retains material tables and Jones exit fields.


In [5]:
# Refine a ±0.5 mm detector strip containing the central beam and satellites.
# The full detector maps above remain 1300 × 1300. Refining this bright strip
# keeps the tutorial practical; it does not bound errors in the faint tails.
check_rows = np.flatnonzero(abs(y_detector)<.5e-3)
full_y, full_ny = y_detector.copy(), ny
try:
    y_detector, ny = full_y[check_rows], len(check_rows)
    refined = detector_pattern(9)
    harmonic_reference = detector_pattern(footprint_samples,cutoff=25)
finally:
    y_detector, ny = full_y, full_ny
baseline = patterns["total"][check_rows]
relative_l1 = np.sum(abs(refined["total"]-baseline))/refined["total"].sum()
print(f"Bright-strip footprint relative L1 difference: {relative_l1:.3%}")
assert relative_l1 < .01, "Increase footprint_samples and the reference sample count."
np.testing.assert_allclose(patterns["total"],patterns["parallel"]+patterns["crossed"],rtol=1e-13)
for image in patterns.values():
    assert image.shape == (1300,1300) and np.isfinite(image).all() and (image>=0).all()
harmonic_l1 = np.sum(abs(harmonic_reference["total"]-baseline))/harmonic_reference["total"].sum()
print(f"Bright-strip harmonic relative L1 difference: {harmonic_l1:.3%}")
assert harmonic_l1 < .01
# Independent midpoint quadrature in both axes, using the SAME stripe series.
# This verifies rotation, completion-of-square phases and normalization.
probe = wavelength*z/period
quad_n = 20000
dq = window_size/quad_n
uv = -window_size/2+(np.arange(quad_n)+.5)*dq
probe_u, probe_v = np.cos(window_angle)*probe, -np.sin(window_angle)*probe
phase_u = np.exp(-1j*np.pi*(probe_u-uv)**2/(wavelength*z))
phase_v = np.exp(-1j*np.pi*(probe_v-uv)**2/(wavelength*z))
numeric = mean_t*(phase_u.sum()*dq)*(phase_v.sum()*dq)
for h in range(1,max_harmonic+1,2):
    for order in (-h,h):
        fu = order*np.cos(window_angle)/period
        fv = -order*np.sin(window_angle)/period
        iu = np.sum(phase_u*np.exp(2j*np.pi*fu*uv))*dq
        iv = np.sum(phase_v*np.exp(2j*np.pi*fv*uv))*dq
        numeric += delta_t*2*(-1)**((h-1)//2)/(np.pi*h)*iu*iv
analytic = rotated_field(np.array([probe]),np.array([0.]),"fresnel",max_harmonic)[0]
np.testing.assert_allclose(numeric,analytic,rtol=.01,atol=1e-16)
# Area and rotated vertex checks for the hard aperture.
assert np.isclose(window_size**2, (np.sqrt(2)*window_size)**2/2)
profile = patterns["crossed"].sum(axis=0)
region = np.flatnonzero(abs(x_detector-probe)<3*wavelength*z/window_size)
measured = x_detector[region[np.argmax(profile[region])]]
assert abs(measured-probe) < 3*pitch
print(f"First crossed peak: {measured*1e3:.3f} mm")
print("Collected photon rate:",patterns["total"].sum())
assert patterns["total"].sum() <= experiment.xray.photon_flux*1.01

with h5py.File(OUT / "square_window_saxs.h5","w") as f:
    f.attrs["experiment_config_json"] = json.dumps(experiment.to_dict(),default=str)
    f.attrs["assumptions"] = "209 nm projected film; abrupt OOP stripes; coherent LV; opaque exterior; no tapered-wall scattering"
    f.attrs["window_angle_rad"] = window_angle
    f.attrs["max_harmonic"] = max_harmonic
    f.attrs["harmonic_relative_l1"] = harmonic_l1
    f.attrs["convergence_region"] = "abs(detector_y) < 0.5 mm; faint tails not bounded"
    f.attrs["footprint_samples"] = footprint_samples
    f.attrs["footprint_relative_l1"] = relative_l1
    f.attrs["wavelength_m"] = wavelength
    f.attrs["intensity_units"] = "photons/s/pixel"
    f.create_dataset("x_detector_m",data=x_detector)
    f.create_dataset("y_detector_m",data=y_detector)
    f.create_dataset("domain_transmissions",data=transmission)
    for model, images in (("fresnel",patterns),("fraunhofer",farfield)):
        for name,image in images.items():
            f.create_dataset(f"{model}/{name}",data=image,compression="gzip")
print("Saved figures and HDF5:",OUT)


Bright-strip footprint relative L1 difference: 0.257%
Bright-strip harmonic relative L1 difference: 0.000%
First crossed peak: 3.190 mm
Collected photon rate: 0.2883773831543043
Saved figures and HDF5: /home/riccardo/Desktop/Github/fomocid/outputs/21_square_window_saxs
